# Bài 17: Nén ảnh &mdash; Không mất mát và Có mất mát (Image Compression &mdash; Lossless and Lossy)

Lưu ảnh dưới dạng điểm ảnh thô tốn 1 byte cho mỗi kênh của mỗi điểm ảnh, bất kể ảnh chứa gì. Nén ảnh khai thác nhiều loại dư thừa khác nhau trong ảnh --- tức là thực tế rằng ảnh thật không phải ngẫu nhiên. Bài này trình bày cả nén **không mất mát (lossless)** (ảnh giải mã giống hệt ảnh gốc đến từng bit &mdash; ví dụ PNG) lẫn nén **có mất mát (lossy)** (ảnh giải mã chỉ là một phép xấp xỉ &mdash; ví dụ JPEG).

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['house.png']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import heapq
from collections import Counter
import matplotlib.pyplot as plt

## Nén không mất mát (Lossless compression)
### Mã hóa loạt dài (RLE) (Run-length encoding (RLE))
Lược đồ không mất mát đơn giản nhất: thay vì lưu mọi điểm ảnh, lưu các cặp `(value, run length)` cho các loạt điểm ảnh giống nhau liên tiếp. Cách này hoạt động rất tốt trên các ảnh có vùng phẳng lớn (đồ họa tổng hợp, văn bản quét, mặt nạ), nhưng chẳng có ích gì với ảnh chụp thật.

In [ ]:
def rle_encode(array):
    flat = array.ravel()
    change_points = np.where(np.diff(flat) != 0)[0] + 1
    starts = np.concatenate([[0], change_points])
    ends = np.concatenate([change_points, [len(flat)]])
    return [(flat[s], e - s) for s, e in zip(starts, ends)]

def rle_decode(runs, shape):
    flat = np.concatenate([np.full(length, value) for value, length in runs])
    return flat.reshape(shape).astype(np.uint8)

flat_regions_img = np.zeros((80, 80), dtype=np.uint8)
cv2.rectangle(flat_regions_img, (10, 10), (70, 70), 200, -1)
sp = flat_regions_img.shape

plt.imshow(flat_regions_img, cmap='gray')
plt.title(f'Image with flat regions ({sp[1]}x{sp[0]}) = {sp[0]*sp[1]} bytes')
plt.axis('off')
plt.show()

runs = rle_encode(flat_regions_img)
decoded = rle_decode(runs, flat_regions_img.shape)
print('exact reconstruction?', np.array_equal(decoded, flat_regions_img))

raw_bytes = flat_regions_img.size
rle_bytes = len(runs) * 3  # roughly: 1 byte value + 2 bytes run length, per run
print(f'raw size:  {raw_bytes} bytes')
print(f'RLE size:  ~{rle_bytes} bytes ({raw_bytes / rle_bytes:.1f}x smaller)')

### Mã hóa Huffman: ít bit hơn cho các giá trị phổ biến (Huffman coding: fewer bits for common values)
Một phương pháp không mất mát tinh vi hơn là **mã hóa Huffman (Huffman coding)**, tận dụng thực tế rằng một số giá trị điểm ảnh (ví dụ một mức xám nền phổ biến) xuất hiện thường xuyên hơn nhiều so với các giá trị khác. Nó xây dựng một bộ mã độ dài thay đổi, trong đó các giá trị thường gặp có mã ngắn và các giá trị hiếm có mã dài hơn &mdash; và đã được chứng minh là tối ưu trong số các mã tiền tố (prefix codes).

**Entropy** của Shannon cho ra cận dưới lý thuyết về số bit trung bình trên mỗi điểm ảnh đối với *bất kỳ* bộ mã nào chỉ dựa trên phân phối giá trị (bỏ qua cấu trúc không gian):

$$H = -\sum_i p_i \log_2 p_i$$

trong đó $p_i$ là xác suất một giá trị điểm ảnh nhất định xuất hiện trong ảnh, và tổng được lấy trên mọi giá trị điểm ảnh có thể.

In [ ]:
rng = np.random.default_rng(0)
photo_like = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(photo_like, (10, 10), (90, 90), 200, -1)
cv2.circle(photo_like, (50, 50), 25, 120, -1)
photo_like = np.clip(photo_like.astype(np.float64) + rng.normal(0, 5, photo_like.shape), 0, 255).astype(np.uint8)
sp = photo_like.shape

counts = Counter(photo_like.ravel().tolist())
total = sum(counts.values())
probs = {value: count / total for value, count in counts.items()}
entropy = -sum(p * np.log2(p) for p in probs.values())


def build_huffman_codes(probs):
    heap = [[p, [symbol, '']] for symbol, p in probs.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
    return {symbol: code for symbol, code in heap[0][1:]}

codes = build_huffman_codes(probs)
avg_bits = sum(probs[symbol] * len(code) for symbol, code in codes.items())

plt.imshow(photo_like, cmap='gray')
plt.title(f'Image with noise ({sp[1]}x{sp[0]}) = {sp[0]*sp[1]} bytes')
plt.axis('off')
plt.show()

print(f'naive fixed-width encoding:           8.00 bits/pixel')
print(f'Shannon entropy (theoretical floor):  {entropy:.2f} bits/pixel')
print(f'Huffman coding achieves:              {avg_bits:.2f} bits/pixel')

Mã hóa Huffman tiến gần đến cận entropy (nó chỉ có thể khớp chính xác khi mọi xác suất tình cờ đều là lũy thừa của 2). Các định dạng không mất mát thực tế như PNG kết hợp một ý tưởng như thế này (mã hóa Huffman/số học) với một **bộ lọc dự đoán (predictive filter)** trước đó &mdash; dự đoán mỗi điểm ảnh từ các điểm lân cận và chỉ mã hóa sai số dự đoán (thường nhỏ), vốn có entropy thấp hơn nhiều so với giá trị điểm ảnh thô.

## Nén có mất mát: DCT (Lossy compression: the DCT)
Ý tưởng cốt lõi của JPEG là tận dụng xu hướng bỏ qua các chi tiết nhỏ của hệ thị giác con người. JPEG chia ảnh thành các khối nhỏ 8x8, rồi biến đổi mỗi khối sang miền tần số để có thể loại bỏ các thành phần tần số cao mà mắt người hầu như không nhận ra. JPEG dùng **biến đổi cosin rời rạc (discrete cosine transform — DCT)**, một họ hàng gần của biến đổi Fourier ở Bài 15. Giống như DFT, DCT biểu diễn lại một khối thành tổng các thành phần tần số &mdash; nhưng chỉ dùng cosin (không có phần ảo) để tránh nhiễu ở biên.

In [ ]:
block = photo_like[6:14, 6:14].astype(np.float64)  # a block straddling the rectangle's sharp edge
dct_block = cv2.dct(block)

energy = dct_block**2
print(f'fraction of block energy in top-left 2x2 coefficients: {energy[:2, :2].sum() / energy.sum():.2%}')
print(f'fraction of block energy in top-left 4x4 coefficients: {energy[:4, :4].sum() / energy.sum():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(block, cmap='gray')
axes[0].set_title('8x8 pixel block')
axes[1].imshow(np.log1p(np.abs(dct_block)), cmap='gray')
axes[1].set_title('log|DCT coefficients|\n(top-left = low frequency)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Lượng tử hóa: vứt bỏ những hệ số ít quan trọng nhất (Quantization: throwing away the coefficients that matter least)
Việc nén xảy ra nhờ **lượng tử hóa (quantizing)** các hệ số DCT &mdash; chia cho một giá trị nào đó rồi làm tròn. Các giá trị dùng để chia được lấy từ một bảng lượng tử hóa, được thiết kế dựa trên độ nhạy thị giác của con người. Nhiều hệ số (đặc biệt là các hệ số tần số cao, vốn mang ít năng lượng nhất) trở thành đúng bằng 0 và hầu như không tốn chỗ lưu trữ.

In [ ]:
def keep_top_left(dct_block, n):
    """Zero out every coefficient outside the top-left nxn corner (crude stand-in for JPEG-style quantization)."""
    masked = np.zeros_like(dct_block)
    masked[:n, :n] = dct_block[:n, :n]
    return masked

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, n in zip(axes, [8, 4, 2, 1]):
    kept = keep_top_left(dct_block, n)
    reconstructed = cv2.idct(kept)
    nonzero = np.count_nonzero(kept)
    ax.imshow(reconstructed, cmap='gray', vmin=block.min(), vmax=block.max())
    ax.set_title(f'{nonzero}/64 coeffs kept', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Ảnh động này cho thấy quá trình tái tạo cùng một khối khi các hệ số được thêm vào lần lượt (theo thứ tự zigzag, bắt đầu từ góc trên-trái tần số thấp) &mdash; hãy quan sát khối sắc nét dần từ một vệt mờ phẳng đến đúng bản gốc khi cả 64 hệ số được giữ lại.

<img src="../animations/dct_reconstruction_demo.gif" alt="Ảnh động tái tạo DCT của một khối 8x8, thêm lần lượt từng hệ số theo thứ tự zigzag bắt đầu từ góc trên-trái tần số thấp, cho thấy khối sắc nét dần từ một xấp xỉ phẳng đến đúng bản gốc" style="display: block; margin: 0 auto; max-width: 680px;">

Chỉ cần giữ góc 2x2 trên-trái (4 trên 64 hệ số &mdash; giảm 16 lần) đã bảo toàn được phần lớn cấu trúc tổng thể của khối; chỉ có chi tiết mịn nhất bị mất.

## JPEG thực tế: sự đánh đổi tốc độ bit-độ méo (Real JPEG: the rate-distortion tradeoff)
Mã hóa JPEG thực tế theo cùng công thức DCT-rồi-lượng-tử-hóa này cho từng khối (với một bảng lượng tử hóa phụ thuộc tần số được thiết kế cẩn thận, cộng với mã hóa loạt dài và Huffman cho các hệ số thưa thu được &mdash; kết hợp các ý tưởng có mất mát và không mất mát trong bài này). Ta dùng trực tiếp bộ mã hóa có sẵn của OpenCV, quét tham số chất lượng, và đo cả kích thước tệp lẫn sai số tái tạo.

In [ ]:
def psnr(original, reconstructed):
    mse = np.mean((original.astype(np.float64) - reconstructed.astype(np.float64))**2)
    return float('inf') if mse == 0 else 10 * np.log10(255**2 / mse)

color_img = np.zeros((200, 200, 3), dtype=np.uint8)
cv2.rectangle(color_img, (20, 20), (180, 180), (255, 120, 30), -1)
cv2.circle(color_img, (100, 100), 60, (30, 200, 255), -1)
color_img = np.clip(color_img.astype(np.float64) + rng.normal(0, 8, color_img.shape), 0, 255).astype(np.uint8)
sp = color_img.shape

qualities = [10, 30, 50, 80, 95, 100]
sizes, psnrs = [], []
for q in qualities:
    ok, encoded = cv2.imencode('.jpg', color_img, [cv2.IMWRITE_JPEG_QUALITY, q])
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    sizes.append(len(encoded))
    psnrs.append(psnr(color_img, decoded))

plt.imshow(color_img)
plt.title(f'Image with noise ({sp[1]}x{sp[0]}) = {sp[0]*sp[1]*sp[2]} bytes')
plt.axis('off')
plt.show()

raw_size = color_img.size
print(f'{"quality":>8} {"file size (bytes)":>18} {"compression ratio":>18} {"PSNR (dB)":>10}')
for q, s, p in zip(qualities, sizes, psnrs):
    print(f'{q:>8} {s:>18} {raw_size / s:>17.1f}x {p:>10.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(qualities, sizes, marker='o')
axes[0].set_xlabel('JPEG quality')
axes[0].set_ylabel('file size (bytes)')
axes[0].set_title('Size vs. quality')

axes[1].plot(sizes, psnrs, marker='o')
axes[1].set_xlabel('file size (bytes)')
axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('Rate-distortion curve')
plt.tight_layout()
plt.show()

Đây là sự đánh đổi cơ bản của nén có mất mát: mỗi byte bạn sẵn lòng bỏ thêm sẽ mua được chất lượng với hiệu quả giảm dần (đường cong tốc độ bit-độ méo (rate-distortion) phẳng dần ở chất lượng cao), và không có bữa trưa miễn phí &mdash; chỉ có lựa chọn đứng ở điểm nào trên đường cong. Với một bức ảnh chụp điển hình, chất lượng JPEG 75 sẽ không nhận ra được trừ khi bạn phóng to, và nó sẽ giảm kích thước tệp 6--8 lần.

## JPEG hay PNG? Chọn đúng định dạng (JPEG or PNG? Choosing the right format)
Các sự đánh đổi ở trên chuyển thành một quy tắc thực tế:

- Dùng **JPEG cho ảnh chụp**. Ảnh chụp chủ yếu gồm các dải chuyển màu mượt &mdash; chính là thứ mà DCT dồn vào một vài hệ số tần số thấp, nên loại bỏ phần còn lại hầu như không thấy được, và JPEG mang lại mức giảm kích thước lớn với tổn thất cảm nhận nhỏ.

- Dùng **PNG cho đồ họa** (ảnh chụp màn hình, sơ đồ, văn bản, biểu tượng, bất cứ thứ gì có vùng phẳng hoặc cạnh sắc). Đồ họa là trường hợp ngược lại: một cạnh sắc trải năng lượng ra *mọi* tần số DCT, nên cùng mức lượng tử hóa mà ảnh chụp chịu được sẽ hiện ra thành hiện tượng rung và vỡ khối thấy rõ quanh các cạnh và chữ. Nén không mất mát xử lý vùng phẳng và cạnh sắc gần như miễn phí (đó chính là điểm mạnh của RLE và bộ lọc dự đoán ở trên), nên với loại nội dung này PNG thường vừa không có nhiễu *vừa* nhỏ hơn JPEG.

In [ ]:
photo = cv2.cvtColor(cv2.imread('../img/house.png'), cv2.COLOR_BGR2RGB)

graphic = np.full((100, 200, 3), 255, dtype=np.uint8)
cv2.rectangle(graphic, (20, 20), (180, 100), (40, 40, 40), -1)
cv2.putText(graphic, 'REPORT', (30, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2)
cv2.line(graphic, (30, 80), (160, 80), (255, 255, 255), 2)

def encode_sizes(image, jpeg_quality=75):
    ok, jpg = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
    ok, png = cv2.imencode('.png', image, [cv2.IMWRITE_PNG_COMPRESSION, 9])
    return jpg, png

jpg_photo, png_photo = encode_sizes(photo)
jpg_graphic, png_graphic = encode_sizes(graphic)

print(f'{"":12} {"Original":>11} {"PNG":>11} {"JPEG (q75)":>11}')
print(f'{"photograph":12} {int(np.prod(photo.shape)):>9} B {len(png_photo):>9} B {len(jpg_photo):>9} B    (PNG is bigger than JPEG by {len(png_photo)/len(jpg_photo):.1f}x)')
print(f'{"graphic":12} {int(np.prod(graphic.shape)):>9} B {len(png_graphic):>9} B {len(jpg_graphic):>9} B    (PNG is smaller than JPEG by {len(png_graphic)/len(jpg_graphic):.1f}x)')

fig, axes = plt.subplots(2, 2, figsize=(7, 7))
axes[0, 0].imshow(photo); axes[0, 0].set_title(f'Photo: original ({int(np.prod(photo.shape))} B)')
axes[0, 1].imshow(cv2.imdecode(jpg_photo, cv2.IMREAD_COLOR)); axes[0, 1].set_title(f'Photo: JPEG q75 ({len(jpg_photo)} B)')
axes[1, 0].imshow(graphic); axes[1, 0].set_title(f'Graphic: original ({int(np.prod(graphic.shape))} B)')
axes[1, 1].imshow(cv2.imdecode(jpg_graphic, cv2.IMREAD_COLOR)); axes[1, 1].set_title(f'Graphic: JPEG q75 ({len(jpg_graphic)} B)')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Ảnh của <a href="https://unsplash.com/@tama66" target="_blank" rel="noopener">Peter Herrmann</a> trên <a href="https://unsplash.com/photos/brown-and-white-concrete-building-eZaEWy2rAIc" target="_blank" rel="noopener">Unsplash</a></p>

### Nhìn kỹ hơn các nhiễu của JPEG (A closer look at JPEG artifacts)
Phóng to vào một vùng có độ tương phản cao (đường mái nhà trên nền trời) ở mức chất lượng thấp hơn nhiều cho thấy trực tiếp cái giá của việc lượng tử hóa mạnh tay: các ô vuông 8x8 vỡ khối và hiện tượng rung quanh cạnh sắc, đúng như kiểu thất bại mô tả ở trên.

In [ ]:
y0, y1, x0, x1 = 10, 60, 50, 150  # roofline against the sky

low_q = 10
jpg_low_q, _ = encode_sizes(photo, jpeg_quality=low_q)
decoded_low_q = cv2.imdecode(jpg_low_q, cv2.IMREAD_COLOR)

zoom = 4
original_crop = cv2.resize(photo[y0:y1, x0:x1], None, fx=zoom, fy=zoom, interpolation=cv2.INTER_NEAREST)
jpeg_crop = cv2.resize(decoded_low_q[y0:y1, x0:x1], None, fx=zoom, fy=zoom, interpolation=cv2.INTER_NEAREST)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(original_crop)
axes[0].set_title('Original crop (zoomed 4x)')
axes[1].imshow(jpeg_crop)
axes[1].set_title(f'JPEG quality {low_q}\n({len(jpg_low_q) / 1024:5.1f} kB)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Ảnh động này quét hệ số chất lượng JPEG từ 5 đến 100 trên cùng vùng cắt &mdash; hãy quan sát các ô vuông vỡ khối và hiện tượng rung quanh đường mái nhà mờ dần khi chất lượng (và kích thước tệp) tăng lên.

<img src="../animations/jpeg_quality_demo.gif" alt="Ảnh động quét chất lượng JPEG từ thấp đến cao trên một vùng cắt phóng to có độ tương phản cao, cho thấy nhiễu vỡ khối và rung mờ dần khi chất lượng tăng" style="display: block; margin: 0 auto; max-width: 680px;">

### Bài tập (Exercises)
1. Chạy thí nghiệm RLE trên `photo_like` (có nhiễu Gauss) thay vì `flat_regions_img`. Nó tạo ra bao nhiêu loạt, và điều đó nói gì về sự phù hợp của RLE với ảnh tự nhiên có nhiễu?
2. `keep_top_left` là một sự thay thế thô sơ có chủ đích cho lượng tử hóa thực sự (vốn thu nhỏ *mọi* hệ số một cách từ từ thay vì đưa cứng một khối hệ số về 0). Hãy thay nó bằng `np.round(dct_block / step) * step` với vài giá trị `step` khác nhau, và so sánh các nhiễu thị giác với phiên bản góc trên-trái.
3. Ở chất lượng JPEG rất thấp, bạn có thể nhận thấy một mẫu lưới thấy rõ thẳng hàng với ranh giới các khối 8x8 ("nhiễu vỡ khối — blocking artifacts"). Dựa trên mối liên hệ giữa ranh giới khối và DCT, hãy giải thích tại sao lượng tử hóa mỗi khối một cách *độc lập* lại tạo ra đúng loại nhiễu này.